# 02 — Building Recommendation-Ready Profiles

Reshapes the cleaned relational tables into one row per learner and one row per course, and builds
the shared skill vocabulary that links the two.

Two data-quality repairs happen here, both discovered after the first pass:

1. **De-duplication of stacked people.** The source population is stacked ~3x under different
   `person_id`s, so `drop_duplicates("person_id")` in notebook 01 could not see it.
2. **Skill-validity filtering.** Roughly a third of "skills" harvested from resumes are actually
   whole bullet points (e.g. *"maintain multiple database environments redshift rds in aws"*).
   These pollute the vocabulary and cannot match anything in the course catalogue.

### Inputs and outputs are kept separate

Reading and writing the same path would let this notebook consume its own input, so a repair could
only ever be demonstrated once. The fallback loader therefore reads a **read-only snapshot** in
`data/source_profiles/` and results are written to `data/recommendation_ready/`.

Each repair reports explicitly when it finds nothing to do, rather than printing a `1.00x`
reduction that could be mistaken for a fresh result.

In [ ]:
import ast
import re
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
CLEAN_DIR = PROJECT_ROOT / "data" / "cleaned"              # normal input (from notebook 01)
SOURCE_DIR = PROJECT_ROOT / "data" / "source_profiles"     # fallback input - READ ONLY
READY_DIR = PROJECT_ROOT / "data" / "recommendation_ready" # output - never read as input

READY_DIR.mkdir(parents=True, exist_ok=True)

# This notebook previously read and wrote the same file, so running it consumed its own
# input and the de-duplication could only ever be demonstrated once. Input and output
# directories are now distinct, and that separation is asserted rather than assumed.
assert SOURCE_DIR != READY_DIR, "fallback input must not be the output directory"


def _as_list(value):
    """Parse a stringified Python list back into a list."""
    if isinstance(value, list):
        return value
    if isinstance(value, str) and value.startswith("["):
        try:
            return ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []
    return []


def load_from_cleaned():
    """Normal path: the outputs of 01_data_cleaning.ipynb."""
    return (
        pd.read_csv(CLEAN_DIR / "people_cleaned.csv"),
        pd.read_csv(CLEAN_DIR / "education_cleaned.csv"),
        pd.read_csv(CLEAN_DIR / "experience_cleaned.csv"),
        pd.read_csv(CLEAN_DIR / "person_skills_cleaned.csv"),
        pd.read_csv(CLEAN_DIR / "courses_cleaned.csv"),
    )


def seed_source_snapshot():
    """One-time copy of published profiles into the read-only fallback directory."""
    SOURCE_DIR.mkdir(parents=True, exist_ok=True)
    for filename in ["user_profiles.csv", "course_profiles.csv"]:
        target = SOURCE_DIR / filename
        if not target.exists():
            shutil.copy2(READY_DIR / filename, target)
            print(f"  seeded {SOURCE_DIR.name}/{filename}")


def rebuild_from_ready():
    """Fallback: reconstruct the long tables from a read-only snapshot of published profiles.

    Lossy - institution/firm/location and raw record counts are not recoverable - but it keeps
    this notebook runnable when only the published profiles survive.
    """
    profiles = pd.read_csv(SOURCE_DIR / "user_profiles.csv")
    for column in ["skills", "experience_titles", "education_programs"]:
        profiles[column] = profiles[column].map(_as_list)

    people = profiles[["person_id", "name"]].copy()
    person_skills = (
        profiles[["person_id", "skills"]]
        .explode("skills")
        .dropna(subset=["skills"])
        .rename(columns={"skills": "skill"})
    )
    experience = (
        profiles[["person_id", "experience_titles"]]
        .explode("experience_titles")
        .dropna(subset=["experience_titles"])
        .rename(columns={"experience_titles": "title"})
    )
    education = (
        profiles[["person_id", "education_programs"]]
        .explode("education_programs")
        .dropna(subset=["education_programs"])
        .rename(columns={"education_programs": "program"})
    )
    courses = pd.read_csv(SOURCE_DIR / "course_profiles.csv")
    return people, education, experience, person_skills, courses


if (CLEAN_DIR / "people_cleaned.csv").exists():
    print("Loading from data/cleaned/ (output of notebook 01).")
    people, education, experience, person_skills, courses = load_from_cleaned()
else:
    print("data/cleaned/ not found - falling back to a read-only profile snapshot.")
    seed_source_snapshot()
    people, education, experience, person_skills, courses = rebuild_from_ready()
    print("Re-run 01_data_cleaning.ipynb from data/raw/ for the full-fidelity pipeline.")

for name, frame in [
    ("people", people), ("education", education), ("experience", experience),
    ("person_skills", person_skills), ("courses", courses),
]:
    print(f"  {name:15} {frame.shape}")

In [2]:
def normalize_skill(skill):
    if pd.isna(skill):
        return None
    
    skill = str(skill).lower().strip()
    
    # Normalize common separators
    skill = skill.replace("&", " and ")
    skill = skill.replace("/", " ")
    skill = skill.replace("-", " ")
    
    # Remove punctuation
    skill = re.sub(r"[^a-z0-9+#.\s]", " ", skill)
    
    # Collapse whitespace
    skill = re.sub(r"\s+", " ", skill).strip()
    
    return skill if skill else None

## Skill-validity filtering

The resume-derived skill column mixes genuine skills with whole resume bullets. The course
catalogue gives us a ground truth for what a real skill string looks like: **every one of the 323
course skills is 1–5 words**, averaging 2.0. We use that shape, plus two lexical rules, to reject
bullets.

The filter is tuned so that it keeps **100% of the 323 known-good course skills** (positive
control) while discarding roughly a third of the user-side vocabulary.

In [3]:
MAX_SKILL_WORDS = 5     # longest genuine skill in the course catalogue
MAX_SKILL_CHARS = 45

# Past-tense/gerund verbs that start a resume bullet rather than name a skill.
_BULLET_VERB = re.compile(
    r"^(managed|designed|implemented|created|maintained|migrated|kept|developed"
    r"|performed|provided|worked|responsible|experience|expertise|assisted"
    r"|supported|handled|ensured|conducted|coordinated|prepared|analyzed|built"
    r"|led|used|using|utilized|involved|participated|assigned|installed"
    r"|configured|monitored|reviewed|generated|executed|deployed|tested|trained"
    r"|wrote|writing|hands\s+on)\b"
)

# Connectives that only appear inside running prose.
_PROSE = re.compile(
    r"\b(in the|to the|and the|of the|for the|with the|on the|as well as"
    r"|such as|based on|according to|able to|responsible for|years? of)\b"
)

_DURATION = re.compile(r"\d+\s*(years?|yrs?)")


def is_valid_skill(skill):
    """True if `skill` looks like a skill name rather than a resume sentence."""
    if not isinstance(skill, str) or not skill.strip():
        return False

    words = skill.split()
    if not 1 <= len(words) <= MAX_SKILL_WORDS:
        return False
    if len(skill) > MAX_SKILL_CHARS:
        return False

    # A leading verb only signals a bullet once the phrase is long enough;
    # "writing" and "managed services" are legitimate skills.
    if len(words) >= 3 and _BULLET_VERB.search(skill):
        return False
    if _PROSE.search(skill):
        return False
    if _DURATION.search(skill):
        return False

    return True

In [4]:
person_skills["skill_normalized"] = (
    person_skills["skill"]
    .apply(normalize_skill)
)

person_skills = person_skills.dropna(
    subset=["skill_normalized"]
)

person_skills = person_skills.drop_duplicates(
    subset=["person_id", "skill_normalized"]
)

In [5]:
# Positive control: the filter must not reject skills the course catalogue actually teaches.
_catalogue_skills = sorted({
    skill
    for row in courses["skills"].fillna("").map(lambda v: str(v).split(","))
    for skill in (normalize_skill(part) for part in row)
    if skill
})

_kept = [skill for skill in _catalogue_skills if is_valid_skill(skill)]
print(f"Positive control: kept {len(_kept)}/{len(_catalogue_skills)} course skills "
      f"({len(_kept) / len(_catalogue_skills) * 100:.1f}%)")

_rejected = [skill for skill in _catalogue_skills if not is_valid_skill(skill)]
if _rejected:
    print("Rejected course skills (should be empty):", _rejected)

Positive control: kept 323/323 course skills (100.0%)


In [ ]:
before_rows = len(person_skills)
before_vocab = person_skills["skill_normalized"].nunique()

person_skills["skill_is_valid"] = person_skills["skill_normalized"].map(is_valid_skill)
person_skills = person_skills[person_skills["skill_is_valid"]].drop(columns=["skill_is_valid"])

after_vocab = person_skills["skill_normalized"].nunique()
print(f"person-skill rows : {before_rows:,} -> {len(person_skills):,}")
print(f"distinct skills   : {before_vocab:,} -> {after_vocab:,}")

if after_vocab == before_vocab:
    print(
        "\nNO-OP: nothing was filtered, so this input has already been through the filter.\n"
        "       The repair itself is unchanged - it simply has no work left to do here.\n"
        "       Run from data/raw/ to see it applied to unfiltered skills."
    )

In [7]:
user_skills = (
    person_skills
    .groupby("person_id")["skill_normalized"]
    .agg(list)
    .reset_index()
    .rename(columns={
        "skill_normalized": "skills"
    })
)

In [8]:
user_skills["skills_text"] = user_skills["skills"].apply(
    lambda x: " ".join(x)
)

In [9]:
user_skills["skill_count"] = (
    user_skills["skills"].apply(len)
)

In [10]:
user_skills.head()

,person_id,skills,skills_text,skill_count
0,1,"[database administration, database, ms sql ser...",database administration database ms sql server...,20
1,2,"[sql server management studio, visual studio, ...",sql server management studio visual studio sql...,17
2,3,"[databases, oracle 10g, sql, linux]",databases oracle 10g sql linux,4
3,5,"[scrum, agile software development, product ba...",scrum agile software development product backl...,27
4,6,"[oracle databases, database administration, da...",oracle databases database administration datab...,36


In [11]:
experience["title_normalized"] = (
    experience["title"]
    .apply(normalize_skill)
)

In [12]:
user_experience = (
    experience
    .dropna(subset=["title_normalized"])
    .groupby("person_id")
    .agg(
        experience_titles=("title_normalized", list),
        experience_count=("title_normalized", "size")
    )
    .reset_index()
)

In [13]:
user_experience["experience_text"] = (
    user_experience["experience_titles"]
    .apply(lambda x: " ".join(x))
)

In [14]:
user_experience.head()

,person_id,experience_titles,experience_count,experience_text
0,1,[database administrator],1,database administrator
1,2,[database administrator],1,database administrator
2,3,[oracle database administrator],1,oracle database administrator
3,4,[amazon redshift administrator and etl develop...,2,amazon redshift administrator and etl develope...
4,5,"[scrum master, oracle database administrator s...",3,scrum master oracle database administrator scr...


In [15]:
user_experience = (
    experience
    .dropna(subset=["title_normalized"])
    .groupby("person_id")
    .agg(
        experience_titles=(
            "title_normalized",
            lambda x: list(dict.fromkeys(x))
        ),
        experience_count=("title_normalized", "size")
    )
    .reset_index()
)

user_experience["experience_text"] = (
    user_experience["experience_titles"]
    .apply(lambda x: " ".join(x))
)

In [16]:
education["program_normalized"] = (
    education["program"]
    .apply(normalize_skill)
)

In [17]:
user_education = (
    education
    .dropna(subset=["program_normalized"])
    .groupby("person_id")
    .agg(
        education_programs=(
            "program_normalized",
            lambda x: list(dict.fromkeys(x))
        ),
        education_count=("program_normalized", "size")
    )
    .reset_index()
)

In [18]:
user_education["education_text"] = (
    user_education["education_programs"]
    .apply(lambda x: " ".join(x))
)

In [19]:
user_profiles = people.copy()

user_profiles = user_profiles.merge(
    user_skills[
        ["person_id", "skills", "skills_text", "skill_count"]
    ],
    on="person_id",
    how="left"
)

user_profiles = user_profiles.merge(
    user_experience[
        [
            "person_id",
            "experience_titles",
            "experience_text",
            "experience_count"
        ]
    ],
    on="person_id",
    how="left"
)

user_profiles = user_profiles.merge(
    user_education[
        [
            "person_id",
            "education_programs",
            "education_text",
            "education_count"
        ]
    ],
    on="person_id",
    how="left"
)

In [20]:
user_profiles["skills"] = (
    user_profiles["skills"]
    .apply(lambda x: x if isinstance(x, list) else [])
)

user_profiles["experience_titles"] = (
    user_profiles["experience_titles"]
    .apply(lambda x: x if isinstance(x, list) else [])
)

user_profiles["education_programs"] = (
    user_profiles["education_programs"]
    .apply(lambda x: x if isinstance(x, list) else [])
)

for col in [
    "skills_text",
    "experience_text",
    "education_text"
]:
    user_profiles[col] = user_profiles[col].fillna("")
    
for col in [
    "skill_count",
    "experience_count",
    "education_count"
]:
    user_profiles[col] = user_profiles[col].fillna(0).astype(int)

In [21]:
user_profiles["profile_text"] = (
    "skills " + user_profiles["skills_text"] +
    " experience " + user_profiles["experience_text"] +
    " education " + user_profiles["education_text"]
)

In [22]:
user_profiles

,person_id,name,skills,skills_text,skill_count,experience_titles,experience_text,experience_count,education_programs,education_text,education_count,profile_text
0,1,Database Administrator,"[database administration, database, ms sql ser...",database administration database ms sql server...,20,[database administrator],database administrator,1,[bachelor of science],bachelor of science,1,skills database administration database ms sql...
1,2,Database Administrator,"[sql server management studio, visual studio, ...",sql server management studio visual studio sql...,17,[database administrator],database administrator,1,[bsc in computer science],bsc in computer science,1,skills sql server management studio visual stu...
2,3,Oracle Database Administrator,"[databases, oracle 10g, sql, linux]",databases oracle 10g sql linux,4,[oracle database administrator],oracle database administrator,1,[master of computer applications in science an...,master of computer applications in science and...,1,skills databases oracle 10g sql linux experien...
3,4,Amazon Redshift Administrator and ETL Develope...,[],,0,[amazon redshift administrator and etl develop...,amazon redshift administrator and etl develope...,2,[bachelor in computer science],bachelor in computer science,1,skills experience amazon redshift administrat...
4,5,Scrum Master Scrum Master Scrum Master,"[scrum, agile software development, product ba...",scrum agile software development product backl...,27,"[scrum master, oracle database administrator s...",scrum master oracle database administrator scr...,3,[],,0,skills scrum agile software development produc...
...,...,...,...,...,...,...,...,...,...,...,...,...
54928,54929,Lead Python Developer,"[django, angular js, javascript, jquery, node....",django angular js javascript jquery node.js py...,82,"[lead python developer, sr. python developer, ...",lead python developer sr. python developer pyt...,4,[],,0,skills django angular js javascript jquery nod...
54929,54930,Full Stack Python Developer,"[python, django, aws, angularjs, bootstrap, ja...",python django aws angularjs bootstrap javascri...,42,"[full stack python developer, sr. python devel...",full stack python developer sr. python develop...,5,[],,0,skills python django aws angularjs bootstrap j...
54930,54931,Eli Lilly,"[python 2.7, html5, css3, ajax, json, jquery, ...",python 2.7 html5 css3 ajax json jquery active ...,110,"[sr. python developer, python developer, java ...",sr. python developer python developer java dev...,3,[],,0,skills python 2.7 html5 css3 ajax json jquery ...
54931,54932,Python Developer,"[python 3.1x, pyquery, pyqt, django, angular.j...",python 3.1x pyquery pyqt django angular.js ope...,47,"[python developer, software developer]",python developer software developer,2,[],,0,skills python 3.1x pyquery pyqt django angular...


In [23]:
user_profiles["career_context"] = (
    user_profiles["experience_titles"]
    .apply(lambda x: " | ".join(x))
)

## De-duplicating stacked people

Every profile in this dataset appears three times under three different `person_id`s — the raw
export is three concatenated copies of the same population (the ids are offset by a constant).
Because the ids differ, the `drop_duplicates("person_id")` in notebook 01 kept all three.

Left uncorrected this would:

* inflate every skill-frequency statistic by 3x, and
* leak the same person into both sides of any train/test split, making offline evaluation
  meaningless.

We fingerprint each learner by content (`name` + `profile_text`) and keep the lowest `person_id`.

In [ ]:
rows_before = len(user_profiles)

fingerprint = (
    user_profiles["name"].fillna("~").astype(str)
    + "||"
    + user_profiles["profile_text"].fillna("~").astype(str)
)

user_profiles = (
    user_profiles.assign(_fingerprint=fingerprint)
    .sort_values("person_id")
    .drop_duplicates(subset=["_fingerprint"], keep="first")
    .drop(columns=["_fingerprint"])
    .reset_index(drop=True)
)

repeat_factor = rows_before / len(user_profiles)
print(f"learner profiles: {rows_before:,} -> {len(user_profiles):,} ({repeat_factor:.2f}x reduction)")

if repeat_factor < 1.05:
    print(
        "\nNO-OP: no duplicates found, so this input has already been de-duplicated.\n"
        "       On the original export this step reported 54,933 -> 18,194 (3.02x);\n"
        "       see PROJECT_LOG.md A2.1. Run from data/raw/ to reproduce it."
    )

# Every downstream table must be restricted to the surviving people.
surviving_ids = set(user_profiles["person_id"])
person_skills = person_skills[person_skills["person_id"].isin(surviving_ids)].copy()

print(f"person-skill rows restricted to survivors: {len(person_skills):,}")

In [25]:
if "unnamed_0" in courses.columns:
    courses = courses.drop(columns=["unnamed_0"])

In [26]:
courses = courses.reset_index(drop=True)

courses["course_id"] = (
    "COURSE_" +
    courses.index.astype(str).str.zfill(4)
)

In [27]:
def parse_course_skills(value):
    if pd.isna(value):
        return []
    
    skills = str(value).split(",")
    
    skills = [
        normalize_skill(skill)
        for skill in skills
    ]
    
    skills = [
        skill for skill in skills
        if skill
    ]
    
    # Remove duplicates while preserving order
    return list(dict.fromkeys(skills))

In [28]:
courses["skills_list"] = (
    courses["skills"]
    .apply(parse_course_skills)
)

In [29]:
courses["skills_normalized"] = (
    courses["skills_list"]
    .apply(lambda x: " ".join(x))
)

In [30]:
courses["skill_count"] = (
    courses["skills_list"].apply(len)
)

In [31]:
courses[
    ["course_id", "title", "skills", "skills_list", "skill_count"]
].head(10)

,course_id,title,skills,skills_list,skill_count
0,COURSE_0000,Google Cybersecurity,"Network Security, Python Programming, Linux, C...","[network security, python programming, linux, ...",14
1,COURSE_0001,Google Data Analytics,"Data Analysis, R Programming, SQL, Business Co...","[data analysis, r programming, sql, business c...",25
2,COURSE_0002,Google Project Management:,"Project Management, Strategy and Operations, L...","[project management, strategy and operations, ...",24
3,COURSE_0003,IBM Data Science,"Python Programming, Data Science, Machine Lear...","[python programming, data science, machine lea...",32
4,COURSE_0004,Google Digital Marketing & E-commerce,"Digital Marketing, Marketing, Marketing Manage...","[digital marketing, marketing, marketing manag...",19
5,COURSE_0005,IBM Data Analyst,"Python Programming, Microsoft Excel, Data Visu...","[python programming, microsoft excel, data vis...",31
6,COURSE_0006,Google IT Support,"Computer Networking, Network Architecture, Net...","[computer networking, network architecture, ne...",19
7,COURSE_0007,Machine Learning,"Machine Learning, Machine Learning Algorithms,...","[machine learning, machine learning algorithms...",16
8,COURSE_0008,Google UX Design,"User Experience, User Experience Design, User ...","[user experience, user experience design, user...",12
9,COURSE_0009,IBM DevOps and Software Engineering,"DevOps, Software Engineering, Cloud Computing,...","[devops, software engineering, cloud computing...",33


In [32]:
courses["title_normalized"] = (
    courses["title"]
    .apply(normalize_skill)
)

courses["organization_normalized"] = (
    courses["organization"]
    .apply(normalize_skill)
)

In [33]:
courses["course_description_clean"] = (
    courses["course_description"]
    .fillna("")
    .astype(str)
)

In [34]:
courses["course_text"] = (
    "title " + courses["title_normalized"].fillna("") +
    " skills " + courses["skills_normalized"].fillna("") +
    " organization " + courses["organization_normalized"].fillna("") +
    " description " + courses["course_description_clean"]
)

In [35]:
courses["skill_text"] = (
    courses["title_normalized"].fillna("") +
    " " +
    courses["skills_normalized"].fillna("")
)

In [36]:
courses["content_text"] = (
    courses["title_normalized"].fillna("") +
    " " +
    courses["skills_normalized"].fillna("") +
    " " +
    courses["course_description_clean"]
)

In [37]:
courses["ratings"] = pd.to_numeric(
    courses["ratings"],
    errors="coerce"
)
courses["review_count"] = pd.to_numeric(
    courses["review_count"],
    errors="coerce"
)
courses["course_students_enrolled"] = pd.to_numeric(
    courses["course_students_enrolled"],
    errors="coerce"
)

In [38]:
print(courses["difficulty"].value_counts(dropna=False))

difficulty
Beginner        295
Intermediate     76
Mixed            18
Advanced         15
Name: count, dtype: int64


In [39]:
courses["difficulty_normalized"] = (
    courses["difficulty"]
    .fillna("unknown")
    .astype(str)
    .str.lower()
    .str.strip()
)

In [40]:
user_skill_vocab = (
    person_skills[
        ["skill_normalized"]
    ]
    .drop_duplicates()
    .rename(columns={
        "skill_normalized": "skill"
    })
)

user_skill_vocab["source"] = "users"

In [41]:
course_skill_vocab = (
    courses[["skills_list"]]
    .explode("skills_list")
    .dropna()
    .rename(columns={
        "skills_list": "skill"
    })
)

course_skill_vocab["source"] = "courses"

In [42]:
skill_vocabulary = pd.concat(
    [
        user_skill_vocab,
        course_skill_vocab
    ],
    ignore_index=True
)

In [43]:
skill_vocabulary = (
    skill_vocabulary
    .drop_duplicates(subset=["skill", "source"])
)

In [44]:
user_skill_frequency = (
    person_skills
    .groupby("skill_normalized")["person_id"]
    .nunique()
    .reset_index()
    .rename(columns={
        "skill_normalized": "skill",
        "person_id": "user_frequency"
    })
)

In [45]:
course_skill_frequency = (
    course_skill_vocab
    .groupby("skill")["skill"]
    .size()
    .reset_index(name="course_frequency")
)

In [46]:
skill_vocabulary = (
    skill_vocabulary
    .drop(columns=["source"])
    .drop_duplicates(subset=["skill"])
    .merge(
        user_skill_frequency,
        on="skill",
        how="left"
    )
    .merge(
        course_skill_frequency,
        on="skill",
        how="left"
    )
)

skill_vocabulary[
    ["user_frequency", "course_frequency"]
] = skill_vocabulary[
    ["user_frequency", "course_frequency"]
].fillna(0).astype(int)

In [47]:
skill_vocabulary.sort_values(
    "user_frequency",
    ascending=False
).head(30)

,skill,user_frequency,course_frequency
1171,javascript,4661,13
1076,html,4058,0
1263,css,3802,0
1262,jquery,3405,0
2232,java,2913,0
9,sql,2881,50
2421,ajax,2639,0
1211,xml,2591,0
251,mysql,2542,0
2965,html5,2527,0


In [48]:
user_skills_set = set(
    user_skill_frequency["skill"]
)

course_skills_set = set(
    course_skill_frequency["skill"]
)

shared_skills = (
    user_skills_set &
    course_skills_set
)

print("Unique user skills:", len(user_skills_set))
print("Unique course skills:", len(course_skills_set))
print("Shared skills:", len(shared_skills))

Unique user skills: 132943
Unique course skills: 323
Shared skills: 237


In [49]:
courses["recommendation_text"] = (
    "title " +
    courses["title_normalized"].fillna("") +
    " skills " +
    courses["skills_normalized"].fillna("")
)

In [ ]:
# Guard: never write back into the directory the fallback loader reads from.
assert SOURCE_DIR != READY_DIR

outputs = [
    ("user_profiles.csv", user_profiles),
    ("course_profiles.csv", courses),
    ("skill_vocabulary.csv", skill_vocabulary),
]

for filename, frame in outputs:
    target = READY_DIR / filename
    try:
        frame.to_csv(target, index=False)
    except PermissionError as exc:
        # Windows locks a file that is open in Excel, and the raw traceback buries the cause.
        raise RuntimeError(
            f"Cannot write {target}.
"
            f"The file is open in another program (Excel locks CSVs against writing).
"
            f"Close it and re-run this cell."
        ) from exc
    print(f"wrote {filename:24} {frame.shape}")

In [51]:
print("USER PROFILES")
print("=" * 50)
print(user_profiles.shape)
print(user_profiles.columns.tolist())
display(user_profiles.head(3))

USER PROFILES
(18194, 13)
['person_id', 'name', 'skills', 'skills_text', 'skill_count', 'experience_titles', 'experience_text', 'experience_count', 'education_programs', 'education_text', 'education_count', 'profile_text', 'career_context']


,person_id,name,skills,skills_text,skill_count,experience_titles,experience_text,experience_count,education_programs,education_text,education_count,profile_text,career_context
0,1,Database Administrator,"[database administration, database, ms sql ser...",database administration database ms sql server...,20,[database administrator],database administrator,1,[bachelor of science],bachelor of science,1,skills database administration database ms sql...,database administrator
1,2,Database Administrator,"[sql server management studio, visual studio, ...",sql server management studio visual studio sql...,17,[database administrator],database administrator,1,[bsc in computer science],bsc in computer science,1,skills sql server management studio visual stu...,database administrator
2,3,Oracle Database Administrator,"[databases, oracle 10g, sql, linux]",databases oracle 10g sql linux,4,[oracle database administrator],oracle database administrator,1,[master of computer applications in science an...,master of computer applications in science and...,1,skills databases oracle 10g sql linux experien...,oracle database administrator


In [52]:
print("\nCOURSE PROFILES")
print("=" * 50)
print(courses.shape)
print(courses.columns.tolist())
display(
    courses[
        [
            "course_id",
            "title",
            "skills",
            "skills_normalized",
            "skill_count",
            "difficulty_normalized"
        ]
    ].head(5)
)


COURSE PROFILES
(404, 23)
['title', 'organization', 'skills', 'ratings', 'course_url', 'course_students_enrolled', 'course_description', 'review_count', 'difficulty', 'type', 'duration', 'skills_normalized', 'course_id', 'skills_list', 'skill_count', 'title_normalized', 'organization_normalized', 'course_description_clean', 'course_text', 'skill_text', 'content_text', 'difficulty_normalized', 'recommendation_text']


,course_id,title,skills,skills_normalized,skill_count,difficulty_normalized
0,COURSE_0000,Google Cybersecurity,"Network Security, Python Programming, Linux, C...",network security python programming linux clou...,14,beginner
1,COURSE_0001,Google Data Analytics,"Data Analysis, R Programming, SQL, Business Co...",data analysis r programming sql business commu...,25,beginner
2,COURSE_0002,Google Project Management:,"Project Management, Strategy and Operations, L...",project management strategy and operations lea...,24,beginner
3,COURSE_0003,IBM Data Science,"Python Programming, Data Science, Machine Lear...",python programming data science machine learni...,32,beginner
4,COURSE_0004,Google Digital Marketing & E-commerce,"Digital Marketing, Marketing, Marketing Manage...",digital marketing marketing marketing manageme...,19,beginner


In [53]:
print("\nSKILL VOCABULARY")
print("=" * 50)
print(skill_vocabulary.shape)
display(
    skill_vocabulary
    .sort_values("user_frequency", ascending=False)
    .head(20)
)


SKILL VOCABULARY
(133029, 3)


,skill,user_frequency,course_frequency
1171,javascript,4661,13
1076,html,4058,0
1263,css,3802,0
1262,jquery,3405,0
2232,java,2913,0
9,sql,2881,50
2421,ajax,2639,0
1211,xml,2591,0
251,mysql,2542,0
2965,html5,2527,0


In [54]:
courses

,title,organization,skills,ratings,course_url,course_students_enrolled,course_description,review_count,difficulty,type,...,skills_list,skill_count,title_normalized,organization_normalized,course_description_clean,course_text,skill_text,content_text,difficulty_normalized,recommendation_text
0,Google Cybersecurity,Google,"Network Security, Python Programming, Linux, C...",4.8,https://www.coursera.org/professional-certific...,NaN,Google Cloud Fundamentals: Core Infrastructure...,NaN,Beginner,Professional Certificate,...,"[network security, python programming, linux, ...",14,google cybersecurity,google,Google Cloud Fundamentals: Core Infrastructure...,title google cybersecurity skills network secu...,google cybersecurity network security python p...,google cybersecurity network security python p...,beginner,title google cybersecurity skills network secu...
1,Google Data Analytics,Google,"Data Analysis, R Programming, SQL, Business Co...",4.8,https://www.coursera.org/professional-certific...,NaN,Prepare for a new career in the high-growth fi...,NaN,Beginner,Professional Certificate,...,"[data analysis, r programming, sql, business c...",25,google data analytics,google,Prepare for a new career in the high-growth fi...,title google data analytics skills data analys...,google data analytics data analysis r programm...,google data analytics data analysis r programm...,beginner,title google data analytics skills data analys...
2,Google Project Management:,Google,"Project Management, Strategy and Operations, L...",4.8,https://www.coursera.org/professional-certific...,NaN,Prepare-se para uma nova carreira no campo de ...,NaN,Beginner,Professional Certificate,...,"[project management, strategy and operations, ...",24,google project management,google,Prepare-se para uma nova carreira no campo de ...,title google project management skills project...,google project management project management s...,google project management project management s...,beginner,title google project management skills project...
3,IBM Data Science,IBM,"Python Programming, Data Science, Machine Lear...",4.6,https://www.coursera.org/professional-certific...,NaN,Prepare for a career in the high-growth field ...,NaN,Beginner,Professional Certificate,...,"[python programming, data science, machine lea...",32,ibm data science,ibm,Prepare for a career in the high-growth field ...,title ibm data science skills python programmi...,ibm data science python programming data scien...,ibm data science python programming data scien...,beginner,title ibm data science skills python programmi...
4,Google Digital Marketing & E-commerce,Google,"Digital Marketing, Marketing, Marketing Manage...",4.8,https://www.coursera.org/professional-certific...,NaN,This course is the eighth course in the Google...,NaN,Beginner,Professional Certificate,...,"[digital marketing, marketing, marketing manag...",19,google digital marketing and e commerce,google,This course is the eighth course in the Google...,title google digital marketing and e commerce ...,google digital marketing and e commerce digita...,google digital marketing and e commerce digita...,beginner,title google digital marketing and e commerce ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399,Digital Manufacturing & Design Technology,University at Buffalo,"Design and Product, Leadership and Management,...",4.6,https://www.coursera.org/specializations/digit...,NaN,Ce cours est conçu pour accompagner toutes les...,NaN,Beginner,Specialization,...,"[design and product, leadership and management...",44,digital manufacturing and design technology,university at buffalo,Ce cours est conçu pour accompagner toutes les...,title digital manufacturing and design technol...,digital manufacturing and design technology de...,digital manufacturing and design technology de...,beginner,title digital manufacturing and design technol...
400,Cybersecurity for Everyone,"University of Maryland, College Park","Cyber